In [ ]:
pip install --upgrade pip

In [ ]:
pip install requests

In [ ]:
pip install BeautifulSoup4

In [ ]:
pip install xgboost lightgbm


In [ ]:
pip install streamlit joblib scikit-learn numpy pandas

In [ ]:
pip install lxml

Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import requests
import lxml
from bs4 import BeautifulSoup

Web Scrapping from Filpkart

In [ ]:
product_name=[]
mobile_prices=[]
des_ram_rom=[]
des_camera=[]
des_battery=[]
mobile_rating=[]

for i in range(430):
    print(i)
    url="https://www.flipkart.com/search?q=mobiles&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page="+str(i)
    r=requests.get(url)
    soup=BeautifulSoup(r.text,"html.parser")
    box=soup.find_all("div",class_="ZFwe0M row")

    for p in box:
        names=p.find_all("div",class_="RG5Slk")
        for i in names:
            name=i.text
            product_name.append(name)
        

        ul_lists = p.find_all("ul", class_="HwRTzP")

        for ul in ul_lists:
            li_tags = ul.find_all("li", class_="DTBslk")
            features = [li.get_text(strip=True) for li in li_tags]
            des_ram_rom.append("")
            des_camera.append("")
            des_battery.append("")
            for xy in features:
                temp = xy.lower()
                if("ram" in temp or "rom" in temp): des_ram_rom[-1] = xy
                if("camera" in temp): des_camera[-1] = xy
                if("battery" in temp): des_battery[-1] = xy

        prices=p.find_all("div",class_="hZ3P6w DeU9vF")
        for i in prices:
            price=i.text
            mobile_prices.append(price)
        
        ratings=p.find_all("div",class_="MKiFS6")
        if len(ratings)==0: mobile_rating.append(0)
        else: mobile_rating.append(ratings[0].text)
        # print(len(product_name),": ",len(mobile_prices),": ",len(mobile_rating),":",len(des_ram_rom),":",len(des_battery),":",len(des_camera))
        

df=pd.DataFrame({"product_name":product_name,"ram_rom":des_ram_rom,"camera":des_camera,"battery":des_battery,"price":mobile_prices,"rating":mobile_rating})
df.to_csv("project_data_filpkart5.csv")


EDA and Data Cleaning

In [36]:
mobile_df= pd.read_csv(r"C:\Users\lenovo\Documents\ML\project_data_filpkart5.csv")
mobile_df.head()

,Unnamed: 0,product_name,ram_rom,camera,battery,price,rating
0,0,"realme P4x 5G (Matte Silver, 128 GB)",6 GB RAM | 128 GB ROM,50MP + 2MP | 8MP Front Camera,7000 mAh Battery,"₹14,999",0.0
1,1,"Samsung Galaxy A35 5G (Awesome Lilac, 128 GB)",8 GB RAM | 128 GB ROM | Expandable Upto 1 TB,50MP + 8MP + 5MP | 13MP Front Camera,5000 mAh Battery,"₹18,499",4.4
2,2,"POCO C71 (Desert Gold, 64 GB)",4 GB RAM | 64 GB ROM | Expandable Upto 2 TB,32MP Rear Camera | 8MP Front Camera,5200 mAh Battery,"₹6,199",3.8
3,3,"POCO C71 (Power Black, 64 GB)",4 GB RAM | 64 GB ROM | Expandable Upto 2 TB,32MP Rear Camera | 8MP Front Camera,5200 mAh Battery,"₹6,199",3.8
4,4,"Samsung Galaxy A35 5G (Awesome Navy, 256 GB)",8 GB RAM | 256 GB ROM | Expandable Upto 1 TB,50MP + 8MP + 5MP | 13MP Front Camera,5000 mAh Battery,"₹20,499",4.4


In [37]:
mobile_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    984 non-null    int64  
 1   product_name  984 non-null    object 
 2   ram_rom       983 non-null    object 
 3   camera        969 non-null    object 
 4   battery       969 non-null    object 
 5   price         984 non-null    object 
 6   rating        984 non-null    float64
dtypes: float64(1), int64(1), object(5)
memory usage: 53.9+ KB


In [ ]:

#  CLEAN product_name
mobile_df["product_name"] = mobile_df["product_name"].str.strip()
mobile_df["product_name"] = mobile_df["product_name"].str.split().str[0].str.upper()
mobile_df = mobile_df.rename(columns={'product_name': 'brand'})


#  CLEAN PRICE
mobile_df["price"] = (
    mobile_df["price"]
    .str.replace("₹", "")
    .str.replace(",", "")
)
mobile_df["price"] = pd.to_numeric(mobile_df["price"], errors="coerce").fillna(0).astype(int)


#  EXTRACT RAM
mobile_df["RAM_GB"] = mobile_df["ram_rom"].str.extract(r'(\d+)\s*GB\s*RAM')
mobile_df["RAM_GB"] = pd.to_numeric(mobile_df["RAM_GB"], errors="coerce").fillna(4).astype(int)


#  EXTRACT ROM
mobile_df["ROM_GB"] = mobile_df["ram_rom"].str.extract(r'RAM \| (\d+)\s*GB')
mobile_df["ROM_GB"] = pd.to_numeric(mobile_df["ROM_GB"], errors="coerce").fillna(64).astype(int)


#  Expandable storage
mobile_df["Expandable_GB"] = mobile_df["ram_rom"].str.extract(r'Expandable Upto (\d+)\s*TB')
mobile_df["Expandable_GB"] = pd.to_numeric(mobile_df["Expandable_GB"], errors="coerce")

# Convert TB → GB
mobile_df["Expandable_GB"] = (mobile_df["Expandable_GB"].fillna(0) * 1024).astype(int)


#  CAMERA: Extract Rear MP
mobile_df["Rear_MP"] = mobile_df["camera"].str.extract(r'(\d+)\s*MP')
mobile_df["Rear_MP"] = pd.to_numeric(mobile_df["Rear_MP"], errors="coerce").fillna(12).astype(int)


#  BATTERY
mobile_df["Battery_mAh"] = mobile_df["battery"].str.extract(r'(\d+)')
mobile_df["Battery_mAh"] = pd.to_numeric(mobile_df["Battery_mAh"], errors="coerce")

# Fill missing values with mean and convert to int
mobile_df["Battery_mAh"] = mobile_df["Battery_mAh"].fillna(int(mobile_df["Battery_mAh"].mean())).astype(int)


#  Drop unnecessary columns
mobile_df = mobile_df.drop(columns=["ram_rom", "camera", "battery", "Unnamed: 0"])


#  Remove duplicates
mobile_df = mobile_df.drop_duplicates()


#  Save cleaned file
mobile_df.to_csv("cleaned_flipkart_data5.csv", index=False)



In [44]:
cleaned_mobile_df=pd.read_csv("cleaned_flipkart_data5.csv")
print(cleaned_mobile_df.head())
print("\n",cleaned_mobile_df.dtypes)
print("\ncount of null values:\n",cleaned_mobile_df.isnull().sum())
print("\nno. of rows:",cleaned_mobile_df.shape)

     brand  price  rating  RAM_GB  ROM_GB  Expandable_GB  Rear_MP  Battery_mAh
0   REALME  14999     0.0       6     128              0       50         7000
1  SAMSUNG  18499     4.4       8     128           1024       50         5000
2     POCO   6199     3.8       4      64           2048       32         5200
3  SAMSUNG  20499     4.4       8     256           1024       50         5000
4    APPLE  69900     4.6       4      64              0       48         5075

 brand             object
price              int64
rating           float64
RAM_GB             int64
ROM_GB             int64
Expandable_GB      int64
Rear_MP            int64
Battery_mAh        int64
dtype: object

count of null values:
 brand            0
price            0
rating           0
RAM_GB           0
ROM_GB           0
Expandable_GB    0
Rear_MP          0
Battery_mAh      0
dtype: int64

no. of rows: (457, 8)


EDA

In [ ]:
cleaned_df = pd.read_csv("cleaned_flipkart_data5.csv")

print(cleaned_df.describe())



# BRAND DISTRIBUTION
plt.figure(figsize=(12,5))
# cleaned_df['brand'].value_counts().plot(kind='bar')
sns.countplot(x=cleaned_df["brand"])
plt.title("Brand Frequency Count")
plt.xlabel("Brand")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


# PRICE DISTRIBUTION
plt.figure(figsize=(10,5))
sns.histplot(cleaned_df["price"], kde=True, bins=20)
plt.title("Price Distribution")
plt.xlabel("Price (₹)")
plt.show()


# RAM DISTRIBUTION
plt.figure(figsize=(10,5))
sns.countplot(x=cleaned_df["RAM_GB"])
plt.title("RAM Size Distribution")
plt.xlabel("RAM (GB)")
plt.show()


# ROM DISTRIBUTION
plt.figure(figsize=(10,5))
sns.countplot(x=cleaned_df["ROM_GB"])
plt.title("ROM Size Distribution")
plt.xlabel("ROM (GB)")
plt.show()


# BATTERY DISTRIBUTION
plt.figure(figsize=(10,5))
sns.histplot(cleaned_df["Battery_mAh"], bins=20, kde=True)
plt.title("Battery Capacity Distribution")
plt.xlabel("Battery (mAh)")
plt.show()


# CORRELATION HEATMAP
numeric_df = cleaned_df.select_dtypes(include=['int64','float64'])

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()



# PRICE vs RAM
plt.figure(figsize=(8,5))
sns.scatterplot(data=cleaned_df, x="RAM_GB", y="price")
plt.title("Price vs RAM")
plt.show()


# PRICE vs ROM
plt.figure(figsize=(8,5))
sns.scatterplot(data=cleaned_df, x="ROM_GB", y="price")
plt.title("Price vs ROM")
plt.show()



# PRICE vs BATTERY
plt.figure(figsize=(8,5))
sns.scatterplot(data=cleaned_df, x="Battery_mAh", y="price")
plt.title("Price vs Battery mAh")
plt.show()



# OUTLIERS (BOX PLOTS)
features = ["price", "RAM_GB", "ROM_GB", "Battery_mAh", "Rear_MP"]

plt.figure(figsize=(15,8))
for i, col in enumerate(features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x=cleaned_df[col])
    plt.title(f"{col} Outliers")
plt.tight_layout()
plt.show()


Preprocessing and ML part

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv("cleaned_flipkart_data5.csv")

X = df[["brand", "RAM_GB", "ROM_GB", "Expandable_GB","Rear_MP", "Battery_mAh", "rating"]]
y = df["price"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

brand_encoded = encoder.fit_transform(X[["brand"]])
# print(brand_encoded)

brand_df = pd.DataFrame(
    brand_encoded,
    columns=encoder.get_feature_names_out(["brand"])
)
# print(brand_df)
X = X.drop("brand", axis=1)
X = pd.concat([X.reset_index(drop=True), brand_df], axis=1)

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()

# Fit and Transform training/testing numeric data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)



# Train 5 models separately
results = []


# LINEAR REGRESSION
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

results.append({
    "Model": "Linear Regression",
    "MAE": round(mean_absolute_error(y_test, y_pred_lr), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_lr)), 2),
    "R² Score": round(r2_score(y_test, y_pred_lr), 3)
})


# RANDOM FOREST
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

results.append({
    "Model": "Random Forest",
    "MAE": round(mean_absolute_error(y_test, y_pred_rf), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_rf)), 2),
    "R² Score": round(r2_score(y_test, y_pred_rf), 3)
})


# GRADIENT BOOSTING
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train_scaled, y_train)
y_pred_gbr = gbr.predict(X_test_scaled)

results.append({
    "Model": "Gradient Boosting",
    "MAE": round(mean_absolute_error(y_test, y_pred_gbr), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_gbr)), 2),
    "R² Score": round(r2_score(y_test, y_pred_gbr), 3)
})


# XGBOOST
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

results.append({
    "Model": "XGBoost",
    "MAE": round(mean_absolute_error(y_test, y_pred_xgb), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_xgb)), 2),
    "R² Score": round(r2_score(y_test, y_pred_xgb), 3)
})


# LIGHTGBM
lgb_model = lgb.LGBMRegressor(random_state=42)
lgb_model.fit(X_train_scaled, y_train)
y_pred_lgb = lgb_model.predict(X_test_scaled)

results.append({
    "Model": "LightGBM",
    "MAE": round(mean_absolute_error(y_test, y_pred_lgb), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_lgb)), 2),
    "R² Score": round(r2_score(y_test, y_pred_lgb), 3)
})


# Comparison Table
results_df = pd.DataFrame(results)
print("\n============ MODEL PERFORMANCE COMPARISON ============\n")
print(results_df)


import joblib

joblib.dump(xgb_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoder, "encoder.pkl")

print("Model and files saved successfully")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000049 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 96
[LightGBM] [Info] Number of data points in the train set: 365, number of used features: 11
[LightGBM] [Info] Start training from score 21873.356164
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

C:\Users\lenovo\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
